In [1]:
from utils import *
import wandb

In [2]:
config = Config().load(os.path.join("configs", "vit.json"))

In [3]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [4]:
# https://github.com/kreasof-ai/sigreg
def sigreg_weak_loss(x, sketch_dim=64):
    """
    Forces Covariance(x) ~ Identity.
    Matches the 2nd Moment (Spherical Cloud).
    """
    N, C = x.size()
    # 1. Sketching (Optional for C=512, but good for consistency)
    if C > sketch_dim:
        S = torch.randn(sketch_dim, C, device=x.device) / (C ** 0.5)
        x = x @ S.T  # [N, sketch_dim]
    else:
        sketch_dim = C

    # 2. Centering & Covariance
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (N - 1 + 1e-6)

    # 3. Target Identity
    target = torch.eye(sketch_dim, device=x.device)

    # 4. Off-diagonal suppression + Diagonal maintenance
    return torch.norm(cov - target, p='fro')


class EmbeddingLoss(nn.Module):
    def __init__(self, temperature=0.04, alpha=0.1):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha

    def forward(self, x, y):
        B = x.shape[0]
        xNorm, yNorm = nn.functional.normalize(x, dim=-1), nn.functional.normalize(y, dim=-1)

        logits = xNorm @ yNorm.T
        logits = logits / self.temperature

        labels = torch.arange(B, device=x.device)

        loss1 = nn.functional.cross_entropy(logits, labels)
        loss2 = nn.functional.cross_entropy(logits.T, labels)

        infoLoss = 0.5 * (loss1 + loss2)

        sigLoss = 0.5 * (sigreg_weak_loss(x) + sigreg_weak_loss(y))

        return {"total": infoLoss + self.alpha * sigLoss, "info": infoLoss, "sig": sigLoss}

In [5]:
def trainModel(config, modelClass, datasetClass):
    dataset = datasetClass(config.dataset, limit=300000)
    model = modelClass(config.model).to(device)

    print(f"Model has {sum([p.numel() for p in model.parameters()])} parameters")

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learningRate)

    objective = EmbeddingLoss()

    train, test = datasetClass.split(dataset, config)
    train = DataLoader(train, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate, num_workers=4)
    test = DataLoader(test, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate, num_workers=4)

    testIter = itertoolsBetter(test)

    run = None
    total = 0

    try:
        for epoch in range(config.epochs):
            progress = 0
            for inputsX, inputsY, info in train:
                model.train()
                optimizer.zero_grad()

                _, outputsX = model(inputsX.to(device))
                _, outputsY = model(inputsY.to(device))
                loss = objective(outputsX, outputsY)

                trainLoss = loss["total"].detach().item()

                loss["total"].backward()
                optimizer.step()

                with torch.no_grad():
                    model.eval()
                    inputsX1, inputsY1, info1 = next(testIter)

                    _, outputsX1 = model(inputsX1.to(device))
                    _, outputsY1 = model(inputsY1.to(device))
                    loss1 = objective(outputsX1, outputsY1)

                    testLoss = loss1["total"].detach().item()

                if run is None:
                    run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                
                run.log({"Train Loss": trainLoss, 
                         "Train InfoNCE": loss["info"].detach().item(),
                         "Train SigREG": loss["sig"].detach().item(),
                         "Test Loss": testLoss,
                         "Test InfoNCE": loss1["info"].detach().item(),
                         "Test SigREG": loss1["sig"].detach().item()
                         }, step=total)

                progress += 1
                total += 1

                print(f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}% |  Train Loss: {trainLoss:.2f} | Test Loss: {testLoss:.2f}", end="")

    except KeyboardInterrupt:
        latestPath = os.path.join("checkpoints", "pretrain", "latest")
        if not os.path.exists(os.path.join("checkpoints", "pretrain", "latest")):
            os.mkdir(latestPath)

        stamp = datetime.now().strftime("%Y-%m-%d %H-%M")
        timePath = os.path.join("checkpoints", "pretrain", stamp)
        if not os.path.exists(timePath):
            os.mkdir(timePath)

        torch.save(model.state_dict(), os.path.join(latestPath, "checkpoint.pt"))
        torch.save(model.state_dict(), os.path.join(timePath, "checkpoint.pt"))
        config.save(os.path.join(latestPath, "config.json"))
        config.save(os.path.join(timePath, "config.json"))
        return model

In [6]:
trainModel(config, ViT, PairedImageData)


Fonts serialized: 1855/3794google\fonts\ofl\kumarone\KumarOne-Regular.ttf execution context too long
Fonts serialized: 2335/3794google\fonts\ofl\notocoloremojicompattest\NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3735/3794google\fonts\ofl\zcoolxiaowei\ZCOOLXiaoWei-Regular.ttf [Errno 22] Invalid argument: 'google\\bitmaps\\????? ?? al.bmp'
Fonts serialized: 3794/3794

24508301 24508301
Model has 5438196 parameters


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\dylan\_netrc.
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


1 | 80416/151264 | 53.163% |  Train Loss: 1.66 | Test Loss: 2.03

OSError: [Errno 22] Invalid argument